In [2]:
import pandas as pd
from gensim.models import Word2Vec

# ───────────────────────────────────────────────
# 1. Load your CSV
# ───────────────────────────────────────────────
csv_path = "wigner_analysis_results_combined.csv"
print("[INFO] Loading CSV...")
df = pd.read_csv(csv_path)

print(f"[INFO] Loaded {len(df)} rows.")

# ───────────────────────────────────────────────
# 2. Extract ground truth text
# ───────────────────────────────────────────────
texts = df["ground_truth"].astype(str).tolist()

# simple tokenizing (split by space)
tokenized = [t.lower().split() for t in texts]

print("[INFO] Tokenized text count:", len(tokenized))

# ───────────────────────────────────────────────
# 3. Train Word2Vec model
# ───────────────────────────────────────────────
print("[INFO] Training Word2Vec...")

model = Word2Vec(
    sentences=tokenized,
    vector_size=300,   # embedding size
    window=5,          # context window
    min_count=1,       # keep all words
    workers=4,         # adjust if needed
    sg=1               # skip-gram (better for small data)
)

print("[INFO] Training done.")

# ───────────────────────────────────────────────
# 4. Save the trained model
# ───────────────────────────────────────────────
out_path = "custom_w2v_groundtruth.model"
model.save(out_path)

print("[INFO] Saved model to:", out_path)


[INFO] Loading CSV...
[INFO] Loaded 10858 rows.
[INFO] Tokenized text count: 10858
[INFO] Training Word2Vec...
[INFO] Training done.
[INFO] Saved model to: custom_w2v_groundtruth.model


In [4]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import time

# ─────────────────────────────────────────────
# 1. Load your trained Word2Vec model
# ─────────────────────────────────────────────
model_path = "custom_w2v_groundtruth.model"
print("[INFO] Loading Word2Vec model...")
model = Word2Vec.load(model_path)
print("[INFO] Model loaded.")

# ─────────────────────────────────────────────
# 2. Load the inference result file
# ─────────────────────────────────────────────
csv_path = "inference_results-qwen-3vl-8b-v2814-v2.csv"
print("[INFO] Loading inference CSV...")
df = pd.read_csv(csv_path)
print("[INFO] Rows:", len(df))

# define your column names:
PRED_COL = "generated"
GT_COL   = "ground_truth"

# ─────────────────────────────────────────────
# 3. Convert sentence to vector
# ─────────────────────────────────────────────
def sentence_vector(text):
    words = str(text).lower().split()
    vecs = [model.wv[w] for w in words if w in model.wv]

    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

# ─────────────────────────────────────────────
# 4. Compute similarity with progress log
# ─────────────────────────────────────────────
scores = []
total = len(df)
last_print = time.time()

print("[INFO] Scoring...")

for i, row in df.iterrows():
    v1 = sentence_vector(row[PRED_COL])
    v2 = sentence_vector(row[GT_COL])

    sim = cosine_similarity([v1], [v2])[0][0]
    scores.append(sim)

    # print progress every second
    if time.time() - last_print > 1:
        pct = (i+1) / total * 100
        print(f"[INFO] {i+1}/{total} ({pct:.2f}%)")
        last_print = time.time()

df["w2v_score"] = scores

print("[INFO] Scoring complete.")

# ─────────────────────────────────────────────
# 5. Save to new CSV
# ─────────────────────────────────────────────
out_path = "inference_results_with_w2v_score.csv"
df.to_csv(out_path, index=False)

print("[INFO] Saved to:", out_path)

# ─────────────────────────────────────────────
# 6. Print mean score
# ─────────────────────────────────────────────
mean_score = df["w2v_score"].mean()
print("[INFO] Mean W2V score:", mean_score)


[INFO] Loading Word2Vec model...
[INFO] Model loaded.
[INFO] Loading inference CSV...
[INFO] Rows: 1086
[INFO] Scoring...
[INFO] 954/1086 (87.85%)
[INFO] Scoring complete.
[INFO] Saved to: inference_results_with_w2v_score.csv
[INFO] Mean W2V score: 0.9854291
